# PULSE — Graph Topology: First Look

Load the persisted relationship graph (post-review, post-derivation) and explore:
- Degree distribution and network statistics
- Top nodes by centrality (bridge nodes, hub nodes)
- Candidate clusters for hidden LP adjacency
- Contradiction hot-spots (high contradiction_score edges)

This is a read-only analysis. Nothing is written back to relationships.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

from agents.graph.persist import load_graph
from agents.graph.metrics import (
    compute_all_metrics, degree_dataframe, top_nodes_by_centrality, contradiction_hotspots
)
from agents.db import get_conn

G = load_graph()
print(f'Loaded graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# Full metrics
metrics = compute_all_metrics(G)
print('Graph metrics:')
for k, v in metrics.items():
    if not isinstance(v, (dict, pd.DataFrame)):
        print(f'  {k}: {v}')

In [ ]:
# Degree distribution
deg_df = degree_dataframe(G)
print(f'\nTop 20 nodes by degree:')
display(deg_df.head(20))

if len(deg_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    deg_df['total_degree'].hist(bins=20, ax=ax)
    ax.set_xlabel('Total Degree')
    ax.set_title('Node Degree Distribution')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top nodes by betweenness centrality (bridge nodes)
top_central = top_nodes_by_centrality(G, n=15)
print('Top bridge nodes (betweenness centrality):')
display(top_central)

In [ ]:
# Contradiction hot-spots
hot_edges = contradiction_hotspots(G, threshold=0.20)
print(f'Contradiction hot-spots (contradiction_score >= 0.20): {len(hot_edges)}')
display(hot_edges.head(20))

In [ ]:
# Edge type distribution
con = get_conn(read_only=True)
edge_types = con.execute("""
    SELECT edge_type, COUNT(*) as cnt, AVG(confidence) as avg_confidence,
           AVG(evidence_count) as avg_evidence_count
    FROM relationships_effective
    GROUP BY edge_type
    ORDER BY cnt DESC
""").fetchdf()
print('Edge type distribution:')
display(edge_types)

In [ ]:
# Low temporal confidence edges (potential stale relationships)
try:
    stale = con.execute("""
        SELECT source_node_id, target_node_id, edge_type,
               relationship_decay_score, temporal_confidence, last_active
        FROM relationships_effective
        WHERE temporal_confidence IS NOT NULL AND temporal_confidence < 0.30
        ORDER BY temporal_confidence ASC
        LIMIT 20
    """).fetchdf()
    print(f'\nPotentially stale edges (temporal_confidence < 0.30): {len(stale)}')
    display(stale)
except Exception as e:
    print(f'No temporal data yet: {e}')